# Module 5: New Intros & Invisible SKUs Price Push

## Purpose
This module runs on schedule to push prices for two categories of SKUs:
1. **New Intros**: Products with stock that have no price set, or invisible packing units
2. **Invisible SKUs**: Products invisible to retailers (deactivated/hidden) but with available stock

## Price Calculation
- Price = WAC × basic_unit_count / (1 - target_margin)
- Target margin priority: brand-category target → category-level fallback → 10% default
- All prices rounded to nearest 0.25 EGP

## Data Sources
- **Snowflake**: Product data, WAC (all_cogs), stocks, visibility, commercial targets
- **MaxAB API**: Pricing upload endpoint (via push_prices_handler)

## Cohorts
700 (Cairo), 701 (Giza), 702 (Alexandria), 703 (Delta West), 704 (Delta East), 1123-1126 (Upper Egypt)

In [1]:
# =============================================================================
# IMPORTS AND SETUP
# =============================================================================
import pandas as pd
import numpy as np
import os
import time
from datetime import datetime
import pytz
import sys
sys.path.append('..')

# queries_module: Snowflake connection + query_snowflake()
%run queries_module.ipynb

# push_prices_handler: API credentials + post_prices()
%run push_prices_handler.ipynb

# Cairo timezone
CAIRO_TZ = pytz.timezone('Africa/Cairo')
CAIRO_NOW = datetime.now(CAIRO_TZ)

# =============================================================================
# CONFIGURATION
# =============================================================================
MODE = 'live'          # 'testing' = prepare only, 'live' = upload to API
UPLOAD_DIR = 'uploads'
DEFAULT_MARGIN = 0.1   # Fallback margin when no target exists

print(f"\n{'='*60}")
print(f"Module 5: New Intros & Invisible SKUs Price Push")
print(f"Run Time (Cairo): {CAIRO_NOW.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {MODE.upper()}")
print(f"{'='*60}")

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/snowflake/connector/options.py:104: UserWarning: You have an incompatible version of 'pyarrow' installed (20.0.0), please install a version that adheres to: 'pyarrow<19.0.0; extra == "pandas"'
  warn_incompatible_dep(


/home/ec2-user/.Renviron
/home/ec2-user/service_account_key.json
Queries Module | Timezone: America/Los_Angeles
✅ UTH and Last Hour functions defined
✅ Yesterday closing stock function defined
Fixed price & cart rule functions defined

QUERIES MODULE READY

Live Data Functions:
  • get_current_stocks()
  • get_packing_units()
  • get_current_prices()
  • get_current_wac()
  • get_current_cart_rules()

UTH Performance Functions:
  • get_uth_performance()         - UTH qty/retailers (Snowflake)
  • get_hourly_distribution()     - Historical hour contributions (Snowflake)
  • get_last_hour_performance()   - Last hour qty/retailers (DWH)

Stock Functions:
  • get_yesterday_closing_stock() - Yesterday closing stock with parent WH mapping

Override Functions:
  • get_fixed_prices()            - Fixed prices from Google Sheet

Note: Market prices use MODULE_1_INPUT data
Quarterly Contribution Query defined ✓
  - get_quarterly_contribution()
Target Turnover Query defined ✓
  - get_target_turno

In [2]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def convert_to_numeric(df):
    """Convert all possible DataFrame columns to numeric types."""
    df.columns = df.columns.str.lower()
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    return df


def round_to_quarter(price):
    """Round price to nearest 0.25 EGP (MaxAB pricing increment)."""
    return np.round(price * 4) / 4

In [3]:
# =============================================================================
# QUERY 1: NEW INTROS
# =============================================================================
# Finds products that have stock but:
#   - No price set in cohort_product_packing_units (price IS NULL)
#   - Invisible packing units with basic_unit_count != 1
#   - Completely invisible across all packing units (any_vis = 0)
# Calculates price using WAC and target margin from commercial_targets

NEW_INTROS_QUERY = '''
WITH mapping AS (
    SELECT *
    FROM (
        VALUES 
            ('Cairo', 'Mostorod', 1, 700),
            ('Giza', 'Barageel', 236, 701),
            ('Giza', 'Sakkarah', 962, 701),
            ('Delta West', 'El-Mahala', 337, 703),
            ('Delta West', 'Tanta', 8, 703),
            ('Delta East', 'Mansoura FC', 339, 704),
            ('Delta East', 'Sharqya', 170, 704),
            ('Upper Egypt', 'Assiut FC', 501, 1124),
            ('Upper Egypt', 'Bani sweif', 401, 1126),
            ('Upper Egypt', 'Menya Samalot', 703, 1123),
            ('Upper Egypt', 'Sohag', 632, 1125),
            ('Alexandria', 'Khorshed Alex', 797, 702)
    ) x(region, warehouse_name, warehouse_id, cohort_id)
),

av_stocks AS (
    WITH parent_whs AS (
        SELECT * 
        FROM (
            VALUES (236,343), (1,467), (962,343)
        ) x(parent_id, child_id)
    )
    SELECT warehouse_id, product_id,
           CASE WHEN stocks_child IS NOT NULL AND stocks = 0 THEN stocks_child ELSE stocks END AS stocks 
    FROM (
        SELECT 
            pw.warehouse_id,
            pw.product_id,
            pw.available_stock::INTEGER AS stocks,
            pw2.available_stock::INTEGER AS stocks_child
        FROM product_warehouse pw
        LEFT JOIN parent_whs p ON p.parent_id = pw.warehouse_id
        LEFT JOIN product_warehouse pw2 ON pw2.warehouse_id = p.child_id AND pw.product_id = pw2.product_id
        WHERE pw.warehouse_id NOT IN (6, 9, 10)
            AND pw.is_basic_unit = 1
    )
),

cat_brand_target AS (
    SELECT DISTINCT cat, brand, margin AS target_bm
    FROM performance.commercial_targets cplan
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', Current_timestamp::date) 
        THEN DATE_TRUNC('month', Current_timestamp::date)
        ELSE DATE_TRUNC('month', Current_timestamp::date - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
),

cat_target AS (
    SELECT cat, SUM(target_bm * (target_nmv / cat_total)) AS cat_target_margin
    FROM (
        SELECT *, SUM(target_nmv) OVER(PARTITION BY cat) AS cat_total
        FROM (
            SELECT cat, brand, AVG(target_bm) AS target_bm, SUM(target_nmv) AS target_nmv
            FROM (
                SELECT DISTINCT date, city AS region, cat, brand, 
                       margin AS target_bm, nmv AS target_nmv
                FROM performance.commercial_targets cplan
                QUALIFY CASE 
                    WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', Current_timestamp::date) 
                    THEN DATE_TRUNC('month', Current_timestamp::date)
                    ELSE DATE_TRUNC('month', Current_timestamp::date - INTERVAL '1 month') 
                END = DATE_TRUNC('month', date)
            ) GROUP BY ALL
        )
    ) GROUP BY ALL
)

SELECT DISTINCT 
    cohort_id, product_id, sku, brand, cat, packing_unit_id, 
    basic_unit_count,
    wac_p * basic_unit_count AS wac,
    COALESCE(COALESCE(target_bm, cat_target_margin), 0.1) AS margin,
    wac / (1 - margin) AS price
FROM (
    SELECT * 
    FROM (
        SELECT DISTINCT 
            pw.warehouse_id,
            m.cohort_id,
            p.id AS product_id,
            CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
            b.name_ar AS brand,
            c.name_ar AS cat,
            pu.packing_unit_id,
            pu.basic_unit_count,
            cpu.price,
            wac1,
            wac_p,
            cbt.target_bm,
            ct.cat_target_margin,
            pw.stocks,
            CASE WHEN cpu.is_visible = 'true' THEN 1 ELSE 0 END AS visiblity,
            MAX(visiblity) OVER(PARTITION BY p.id, m.cohort_id) AS any_vis 
        FROM products p
        JOIN packing_unit_products pu ON pu.product_id = p.id 
        JOIN product_units ON product_units.id = p.unit_id 
        JOIN finance.all_cogs f ON f.product_id = p.id AND CURRENT_TIMESTAMP BETWEEN from_date AND to_date 
        JOIN brands b ON b.id = p.brand_id
        JOIN categories c ON c.id = p.category_id
        JOIN av_stocks pw ON pw.product_id = p.id 
        JOIN mapping m ON m.warehouse_id = pw.warehouse_id
        LEFT JOIN cohort_product_packing_units cpu ON cpu.cohort_id = m.cohort_id AND cpu.product_packing_unit_id = pu.id
        LEFT JOIN cat_brand_target cbt ON cbt.cat = c.name_ar AND cbt.brand = b.name_ar 
        LEFT JOIN cat_target ct ON ct.cat = c.name_ar 
        WHERE pw.warehouse_id NOT IN (6, 9, 10)
            AND pw.stocks > 0
            AND wac1 > 1 
            AND pu.deleted_at IS NULL
            AND c.name_ar NOT LIKE '%سايب%'
            AND c.name_ar NOT LIKE '%بالتة%'
    )
    WHERE (price IS NULL OR (visiblity = 0 AND basic_unit_count <> 1) OR (visiblity = 0 AND any_vis = 0))
)
'''

print("Fetching new intros from Snowflake...")
new_intros = query_snowflake(NEW_INTROS_QUERY)
new_intros = convert_to_numeric(new_intros)
print(f"  New Intros: {len(new_intros)} records")
if not new_intros.empty:
    print(f"  Cohorts: {sorted(new_intros.cohort_id.unique())}")
    print(f"  Unique products: {new_intros.product_id.nunique()}")
new_intros.head()

Fetching new intros from Snowflake...
  New Intros: 0 records


/tmp/ipykernel_7958/1836834576.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,cohort_id,product_id,sku,brand,cat,packing_unit_id,basic_unit_count,wac,margin,price


In [4]:
# =============================================================================
# QUERY 2: INVISIBLE SKUs
# =============================================================================
# Finds SKUs that are invisible to retailers but have available stock.
# Checks visibility through the full cohort pricing chain (general → main → custom).
# Filters: stock > 0, invisible, not excluded categories, wac_p > 0
# Calculates price using WAC and target margin (embedded in query)

INVISIBLE_QUERY = '''
WITH
cat_brand_target AS (
    SELECT DISTINCT cat, brand, margin AS target_bm
    FROM performance.commercial_targets cplan
    QUALIFY CASE 
        WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', Current_timestamp::date) 
        THEN DATE_TRUNC('month', Current_timestamp::date)
        ELSE DATE_TRUNC('month', Current_timestamp::date - INTERVAL '1 month') 
    END = DATE_TRUNC('month', date)
),

cat_target AS (
    SELECT cat, SUM(target_bm * (target_nmv / cat_total)) AS cat_target_margin
    FROM (
        SELECT *, SUM(target_nmv) OVER(PARTITION BY cat) AS cat_total
        FROM (
            SELECT cat, brand, AVG(target_bm) AS target_bm, SUM(target_nmv) AS target_nmv
            FROM (
                SELECT DISTINCT date, city AS region, cat, brand, 
                       margin AS target_bm, nmv AS target_nmv
                FROM performance.commercial_targets cplan
                QUALIFY CASE 
                    WHEN DATE_TRUNC('month', MAX(DATE) OVER()) = DATE_TRUNC('month', Current_timestamp::date) 
                    THEN DATE_TRUNC('month', Current_timestamp::date)
                    ELSE DATE_TRUNC('month', Current_timestamp::date - INTERVAL '1 month') 
                END = DATE_TRUNC('month', date)
            ) GROUP BY ALL
        )
    ) GROUP BY ALL
),

pw AS (
    SELECT DISTINCT 
        product_warehouse.product_id, 
        product_warehouse.packing_unit_id, 
        pup.id AS product_packing_unit_id, 
        warehouse_id, 
        product_warehouse.available_stock, 
        product_warehouse.stock, 
        product_warehouse.activation AS pw_activation, 
        p.activation AS p_activation, 
        product_warehouse.is_basic_unit, 
        basic_unit_count
    FROM public.product_warehouse
    LEFT JOIN products p ON p.id = product_warehouse.product_id
    INNER JOIN packing_unit_products pup 
        ON pup.product_id = product_warehouse.product_id 
        AND pup.packing_unit_id = product_warehouse.packing_unit_id
),

retailer_cohorts AS (
    SELECT * 
    FROM (
        SELECT DISTINCT
            taggable_id,
            CASE WHEN custom_ct = 1 THEN cohort_id END AS custom_cohort,
            CASE WHEN custom_ct = 1 THEN fallbk ELSE cohort_id END AS main_cohort,
            61 AS general_cohort
        FROM (
            SELECT 
                taggable_id,
                cohorts.id AS cohort_id,
                fallback_cohort_id AS fallbk,
                cohorts.priority,
                CASE WHEN COALESCE(fallback_cohort_id, 61) = 61 THEN 0 ELSE 1 END AS custom_ct,
                RANK() OVER (PARTITION BY taggable_id, custom_ct ORDER BY priority) AS rk,
                SUM(custom_ct) OVER (PARTITION BY taggable_id) AS custom_count
            FROM cohorts
            LEFT JOIN dynamic_taggables AS dt ON dt.dynamic_tag_id = cohorts.dynamic_tag_id
            WHERE cohorts.is_active = true AND cohorts.id <> 61 
        ) AS sub 
        WHERE rk = 1 AND (custom_count = 0 OR (custom_count > 0 AND custom_ct = 1))
        ORDER BY 1
    )
    QUALIFY ROW_NUMBER() OVER(PARTITION BY custom_cohort, main_cohort, general_cohort ORDER BY taggable_id) <= 3000
),

dr AS (
    SELECT DISTINCT 
        retailer_id, product_id, warehouse_id
    FROM warehouse_dispatching_rules AS wdr
    LEFT JOIN (
        SELECT * FROM (
            SELECT *, ROW_NUMBER() OVER (PARTITION BY retailer_id ORDER BY polygon_id) AS r 
            FROM materialized_views.retailer_polygon
        ) AS t1 WHERE r = 1
    ) AS retailer_polygon ON retailer_polygon.polygon_id = wdr.dispatching_polygon_id
),

invisible_raw AS (
    SELECT DISTINCT 
        custom_cohort AS cohort_id, product_id, packing_unit_id, 
        sku, brand, cat, r_price AS current_price, 
        wac_p, wac1, basic_unit_count
    FROM (
        SELECT s.*, wac_p, wac1,
               COALESCE(pv.wac1 * stock, 0) AS stock_value
        FROM (
            SELECT DISTINCT 
                warehouse_id, w.name AS warehouse_name, 
                custom_cohort, main_cohort, general_cohort, 
                product_id, product_packing_unit_id, packing_unit_id,
                CONCAT(p.name_ar, ' ', p.size, ' ', product_units.name_ar) AS sku,
                b.name_ar AS brand,
                c.name_ar AS cat,
                s.stock, s.available_stock, 
                p_activation, pw_activation, 
                CASE WHEN v2 = 0 THEN false ELSE true END AS visible, 
                s.basic_unit_count, r_price
            FROM (
                SELECT *, 
                    CASE WHEN visible = true THEN 1 ELSE 0 END AS v1, 
                    MAX(v1) OVER(PARTITION BY retailer_id, product_id) AS v2
                FROM (
                    SELECT DISTINCT 
                        dr.*,
                        pw.product_packing_unit_id,
                        pw.packing_unit_id,
                        custom_cohort, main_cohort, general_cohort, 
                        pw.stock, pw.available_stock, 
                        basic_unit_count, pw.is_basic_unit, 
                        p_activation, pw_activation, 
                        COALESCE(
                            CASE WHEN cppu3.is_customized = true THEN cppu3.is_visible END, 
                            cppu2.is_visible, cppu.is_visible
                        ) AS visible,
                        COALESCE(
                            CASE WHEN cppu3.is_customized = true THEN cppu3.price END, 
                            cppu2.price, cppu.price
                        ) AS r_price
                    FROM dr 
                    INNER JOIN retailer_cohorts rc ON rc.taggable_id = dr.retailer_id 
                    INNER JOIN pw ON pw.product_id = dr.product_id AND pw.warehouse_id = dr.warehouse_id
                    LEFT JOIN cohort_product_packing_units AS cppu 
                        ON cppu.cohort_id = rc.general_cohort AND cppu.product_packing_unit_id = pw.product_packing_unit_id
                    LEFT JOIN cohort_product_packing_units AS cppu2 
                        ON cppu2.cohort_id = rc.main_cohort AND cppu2.product_packing_unit_id = cppu.product_packing_unit_id
                    LEFT JOIN cohort_product_packing_units AS cppu3 
                        ON cppu3.cohort_id = rc.custom_cohort AND cppu3.product_packing_unit_id = cppu.product_packing_unit_id
                ) s
            ) s 
            JOIN products p ON p.id = s.product_id 
            JOIN categories c ON c.id = p.category_id
            JOIN brands b ON b.id = p.brand_id
            JOIN product_units ON product_units.id = p.unit_id 
            LEFT JOIN warehouses w ON w.id = s.warehouse_id 
            WHERE s.stock > 0 
                AND (pw_activation <> true OR p_activation <> true OR v2 = 0)
        ) s 
        JOIN finance.all_cogs pv ON pv.product_id = s.product_id AND CURRENT_TIMESTAMP BETWEEN from_date AND to_date 
        WHERE s.available_stock > 0  
            AND visible = false 
            AND cat NOT LIKE '%سايب%'
            AND sku NOT LIKE '%@#%' 
            AND sku NOT LIKE '%الكنز%' 
            AND sku NOT LIKE '%سحب%' 
            AND cat NOT LIKE '%بالتة%'
            AND brand NOT LIKE '%مكسب%'
            AND custom_cohort IN (700,701,702,703,704,1123,1124,1125,1126)
            AND wac_p > 0 
    )
)

SELECT 
    ir.cohort_id, ir.product_id, ir.packing_unit_id, ir.sku, ir.brand, ir.cat,
    ir.wac_p, ir.basic_unit_count,
    COALESCE(cbt.target_bm, ct.cat_target_margin, 0.1) AS margin,
    ir.wac_p * ir.basic_unit_count / (1 - COALESCE(cbt.target_bm, ct.cat_target_margin, 0.1)) AS price
FROM invisible_raw ir
LEFT JOIN cat_brand_target cbt ON cbt.cat = ir.cat AND cbt.brand = ir.brand
LEFT JOIN cat_target ct ON ct.cat = ir.cat
WHERE ir.wac_p > 0
'''

print("Fetching invisible SKUs from Snowflake...")
invisible = query_snowflake(INVISIBLE_QUERY)
invisible = convert_to_numeric(invisible)
print(f"  Invisible SKUs: {len(invisible)} records")
if not invisible.empty:
    print(f"  Cohorts: {sorted(invisible.cohort_id.unique())}")
    print(f"  Unique products: {invisible.product_id.nunique()}")
invisible.head()

Fetching invisible SKUs from Snowflake...
  Invisible SKUs: 29 records
  Cohorts: [700, 702, 703, 704, 1123, 1125]
  Unique products: 22


/tmp/ipykernel_7958/1836834576.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


,cohort_id,product_id,packing_unit_id,sku,brand,cat,wac_p,basic_unit_count,margin,price
0,702,26894,7,ايبون فروتي- 620 جرام,ايبون,حلويات و لبان,102.592400,1,0.049263,107.908282
1,703,26391,3,تورتا ديلايتس لاتيه - 10 جنية,نوفي,كيك وكرواسون,44.180000,1,0.080000,48.021739
2,700,26553,1,ثمرات زيت خليط -. 1.7 لتر,ثمرات,زيوت,523.390805,1,0.050000,550.937689
3,703,25993,80,حفاضات بى بم كومفورت مقاس 3 - 35 حفاضة,بي بم,حفاضات أطفال,162.960838,1,0.060000,173.362593
4,702,26394,3,ماجيك فينجرز قطع الكاكاو بالشوكولاته - 10 جنية,نوفي,ويفر,44.180000,1,0.066787,47.341810


In [5]:
# =============================================================================
# PROCESS AND COMBINE
# =============================================================================

# --- Process New Intros ---
to_upload_new = pd.DataFrame()
if not new_intros.empty:
    to_upload_new = new_intros.copy()
    to_upload_new['final_new_price'] = round_to_quarter(to_upload_new['price'])
    to_upload_new = to_upload_new[['cohort_id', 'product_id', 'sku', 'packing_unit_id', 'final_new_price']]
    to_upload_new['source'] = 'new_intro'

# --- Process Invisible SKUs ---
to_upload_inv = pd.DataFrame()
if not invisible.empty:
    to_upload_inv = invisible.copy()
    to_upload_inv['final_new_price'] = round_to_quarter(to_upload_inv['price'])
    # Deduplicate: same product+cohort+PU may appear from multiple retailers/warehouses → take max price
    to_upload_inv = to_upload_inv.groupby(
        ['cohort_id', 'product_id', 'sku', 'packing_unit_id']
    )['final_new_price'].max().reset_index()
    to_upload_inv['source'] = 'invisible'

# --- Combine both datasets ---
to_upload = pd.concat([to_upload_new, to_upload_inv], ignore_index=True)

# If a product appears in both new_intros and invisible, keep the new_intro record
to_upload = to_upload.drop_duplicates(
    subset=['cohort_id', 'product_id', 'packing_unit_id'], keep='first'
)

# Add upload metadata columns
to_upload['ind'] = 1
to_upload['remove_min'] = np.nan

# Filter out invalid prices
to_upload = to_upload[to_upload['final_new_price'] > 1].reset_index(drop=True)

print(f"\n{'='*60}")
print(f"PROCESSING SUMMARY")
print(f"{'='*60}")
print(f"  New Intros:      {len(to_upload_new)} rows")
print(f"  Invisible SKUs:  {len(to_upload_inv)} rows")
print(f"  Combined total:  {len(to_upload)} rows (after dedup)")
if not to_upload.empty:
    print(f"  Cohorts:         {sorted(to_upload.cohort_id.unique())}")
    print(f"  Unique products: {to_upload.product_id.nunique()}")
    print(f"\n  By source:")
    print(to_upload.groupby('source').size().to_string())
to_upload.head(10)


PROCESSING SUMMARY
  New Intros:      0 rows
  Invisible SKUs:  29 rows
  Combined total:  24 rows (after dedup)
  Cohorts:         [700, 702, 703, 704]
  Unique products: 21

  By source:
source
invisible    24


,cohort_id,product_id,sku,packing_unit_id,final_new_price,source,ind,remove_min
0,700,24706,ماكسي اناناس - 2 لتر,2,159.25,invisible,1,NaN
1,700,25538,%تايجر اكسلانس ليمون حلو زياده 15 - 20 جنية,1,181.50,invisible,1,NaN
2,700,25909,فلاي فود صلصة ظرف 5 جنيه - 24 ظرف,3,90.75,invisible,1,NaN
3,700,25991,حفاضات بى بم كومفورت مقاس 1 - 35 حفاضة,80,147.25,invisible,1,NaN
4,700,26388,تورتا رولز شوكولاتة مغطي ومحشو بالشوكولاتة - 1...,3,48.00,invisible,1,NaN
5,700,26395,سباجيتوس اكسترا كولا- شعرية حلوة بطعم الكولا -...,3,28.50,invisible,1,NaN
6,700,26553,ثمرات زيت خليط -. 1.7 لتر,1,551.00,invisible,1,NaN
7,700,26653,بقسماط اكسترا كريسبى - 120 جم,1,390.00,invisible,1,NaN
8,702,25644,الاميرة رول فويل 5 م - 40 سم,104,17.50,invisible,1,NaN
9,702,25658,ترافل جولد شيكولاتة بطعم البندق - 1 كيلو,3,220.75,invisible,1,NaN


In [6]:
# =============================================================================
# UPLOAD TO API
# =============================================================================
os.makedirs(UPLOAD_DIR, exist_ok=True)

if to_upload.empty:
    print("No prices to upload. Exiting.")
else:
    results = {'success': [], 'failed': []}
    
    for cohort in sorted(to_upload.cohort_id.unique()):
        cohort_data = to_upload[to_upload['cohort_id'] == cohort].copy()
        
        # Format for MaxAB API
        out = cohort_data[['product_id', 'sku', 'packing_unit_id', 'final_new_price', 'ind', 'remove_min']].copy()
        out.columns = ['Product ID', 'Product Name', 'Packing Unit ID', 'Price', 'ind', 'remove_min']
        
        # Visibility: YES by default, NO if ind=1 and remove_min=1
        out['Visibility (YES/NO)'] = 'YES'
        out.loc[(out['ind'] == 1) & (out['remove_min'] == 1), 'Visibility (YES/NO)'] = 'NO'
        
        out = out.drop(columns=['ind', 'remove_min']).drop_duplicates()
        out['Execute At (format:dd/mm/yyyy HH:mm)'] = None
        out['Tags'] = None
        
        # Filter out invalid prices
        out = out[out['Price'] > 1].reset_index(drop=True)
        
        if len(out) == 0:
            continue
        
        file_name = f'{UPLOAD_DIR}/new_intros_invisible_{cohort}.xlsx'
        out.to_excel(file_name, index=False, engine='xlsxwriter')
        
        # Testing mode: prepare files but don't upload
        if MODE == 'testing':
            print(f"  [TESTING] Cohort {cohort}: {len(out)} prices prepared (not uploaded)")
            results['success'].append(cohort)
            continue
        
        # Live mode: upload to API
        time.sleep(5)
        response = post_prices(cohort, file_name)
        
        if '"success":true' in str(response.content).lower():
            print(f"  ✓ Cohort {cohort}: {len(out)} prices uploaded successfully")
            results['success'].append(cohort)
        else:
            print(f"  ✗ Cohort {cohort}: Upload FAILED")
            print(f"    Response: {response.content}")
            results['failed'].append(cohort)
            break
    
    # Final summary
    print(f"\n{'='*60}")
    print(f"UPLOAD COMPLETE {'(TESTING MODE)' if MODE == 'testing' else ''}")
    print(f"{'='*60}")
    print(f"  Succeeded: {len(results['success'])} cohorts {results['success']}")
    print(f"  Failed:    {len(results['failed'])} cohorts {results['failed']}")
    print(f"  Timestamp: {datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M:%S')} Cairo")

  ✓ Cohort 700: 8 prices uploaded successfully
  ✓ Cohort 702: 5 prices uploaded successfully
  ✓ Cohort 703: 10 prices uploaded successfully
  ✓ Cohort 704: 1 prices uploaded successfully

UPLOAD COMPLETE 
  Succeeded: 4 cohorts [700, 702, 703, 704]
  Failed:    0 cohorts []
  Timestamp: 2026-03-25 15:00:07 Cairo
